# Download Book Cover Images Locally

This notebook reads `data/books.json`, downloads each book's cover image from the remote CDN URL, and saves it locally under `image/covers/{genre}/`. It then updates `books.json` with a `local_cover` field pointing to the relative local path so the HTML pages can display images offline.

In [1]:
# Cell 1: Import Required Libraries
import json
import os
import time
import requests
from pathlib import Path
from urllib.parse import urlparse

# Project root (one level up from notebook/)
ROOT = Path(os.getcwd()).parent
DATA_DIR = ROOT / "data"
COVERS_DIR = ROOT / "image" / "covers"

print(f"Project root: {ROOT}")
print(f"Data dir:     {DATA_DIR}")
print(f"Covers dir:   {COVERS_DIR}")

Project root: c:\Users\khala\MyDocuments\00_DS\03_CourseWork\2603522_RecSys\100_FinalProject_RecSys
Data dir:     c:\Users\khala\MyDocuments\00_DS\03_CourseWork\2603522_RecSys\100_FinalProject_RecSys\data
Covers dir:   c:\Users\khala\MyDocuments\00_DS\03_CourseWork\2603522_RecSys\100_FinalProject_RecSys\image\covers


In [2]:
# Cell 2: Load books.json
with open(DATA_DIR / "books.json", "r", encoding="utf-8") as f:
    books = json.load(f)

print(f"Loaded {len(books)} books")
print(f"Sample: {books[0]['title_th']} — {books[0]['cover_url'][:60]}...")

Loaded 115 books
Sample: LEAN คิดแบบผู้นำ ทำแบบคนสำเร็จ — https://cdn-shop.ookbee.com/Books/KN19119GMAILCOM/2020/20200...


In [3]:
# Cell 3: Download cover images locally
COVERS_DIR.mkdir(parents=True, exist_ok=True)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
}

downloaded = 0
skipped = 0
failed = 0

for book in books:
    genre = book.get("genre", "other")
    book_id = book.get("book_id", "unknown")
    cover_url = book.get("cover_url", "")

    # Create genre subfolder
    genre_dir = COVERS_DIR / genre
    genre_dir.mkdir(parents=True, exist_ok=True)

    # Determine file extension from URL
    ext = Path(urlparse(cover_url).path).suffix or ".jpg"
    local_filename = f"{book_id}{ext}"
    local_path = genre_dir / local_filename
    relative_path = f"image/covers/{genre}/{local_filename}"

    # Skip if already downloaded
    if local_path.exists():
        book["local_cover"] = relative_path
        skipped += 1
        continue

    # Download
    if not cover_url:
        book["local_cover"] = ""
        failed += 1
        continue

    try:
        resp = requests.get(cover_url, headers=headers, timeout=15)
        resp.raise_for_status()
        local_path.write_bytes(resp.content)
        book["local_cover"] = relative_path
        downloaded += 1
        time.sleep(0.3)  # Be respectful
    except Exception as e:
        print(f"  FAILED {book_id}: {e}")
        book["local_cover"] = ""
        failed += 1

print(f"\nDone! Downloaded: {downloaded}, Skipped (exists): {skipped}, Failed: {failed}")


Done! Downloaded: 20, Skipped (exists): 0, Failed: 95


In [4]:
# Cell 4: Save updated books.json with local_cover paths
with open(DATA_DIR / "books.json", "w", encoding="utf-8") as f:
    json.dump(books, f, ensure_ascii=False, indent=2)

print(f"Saved {len(books)} books to {DATA_DIR / 'books.json'}")

# Verify
with_covers = sum(1 for b in books if b.get("local_cover"))
print(f"Books with local covers: {with_covers}/{len(books)}")

Saved 115 books to c:\Users\khala\MyDocuments\00_DS\03_CourseWork\2603522_RecSys\100_FinalProject_RecSys\data\books.json
Books with local covers: 20/115


In [5]:
# Cell 5: Verify output — list files per genre
for genre_dir in sorted(COVERS_DIR.iterdir()):
    if genre_dir.is_dir():
        count = len(list(genre_dir.glob("*")))
        print(f"  {genre_dir.name}: {count} images")

  art: 0 images
  biography: 0 images
  business: 0 images
  classics: 0 images
  comics: 0 images
  family: 0 images
  fantasy: 0 images
  food: 0 images
  general: 0 images
  health: 0 images
  history: 0 images
  language: 0 images
  mystery: 0 images
  psychology: 20 images
  religion: 0 images
  romance: 0 images
  scifi: 0 images
  tech: 0 images
  travel: 0 images
  ya: 0 images
